In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
from mvlearn.embed import MCCA

# Import split train test from sklearn
from sklearn.model_selection import train_test_split

# Also scipy function for learn a procrustes transformation
from scipy.spatial import procrustes

from fmri_mapping.embedding.evaluation import accuracy_cosine_similarity, compute_rsa
from fmri_mapping.embedding.ops import split_repetitions


from fmri_mapping.embedding.evaluation import compute_rsa, accuracy_cosine_similarity
from fmri_mapping.embedding.ops import split_repetitions
from fmri_mapping.io.nsd import get_resource
from tqdm.notebook import trange
import torch

import numpy as np
import scipy as sp
import ot
from scipy.optimize import linear_sum_assignment
from sklearn.metrics.pairwise import cosine_similarity



In [6]:

def get_common_nsd_id_stimuli(min_reps: int = 3):
    df = get_resource("stimulus").query(
        "shared and exists and (repetition == @min_reps - 1)"
    )
    df_count_subjects = df.groupby("nsd_id").subject.count().reset_index()
    df_count_subjects = df_count_subjects[
        df_count_subjects.subject == 8
    ]  # All subjects

    return list(df_count_subjects.nsd_id.unique())


test_stimuli = get_common_nsd_id_stimuli()


def raw_fmri_avg(subject: int, template_file: str):
    df_reps = split_repetitions(subject=subject)
    df_reps = df_reps.query("nsd_id in @test_stimuli").reset_index(drop=True)


    X = np.load(template_file.format(subject=subject))

    X1 = X[df_reps.subject_index_1]
    X2 = X[df_reps.subject_index_2]
    X3 = X[df_reps.subject_index_3]
    X_avg = (X1 + X2 + X3) / 3

    return X_avg




def learn_mca(X, Y, X_test, Y_test, n_components=128):
    mcca = MCCA(n_components=n_components)
    mcca.fit([X, Y])
    X_transformed, Y_transformed = mcca.transform([X, Y])
    return X_transformed, Y_transformed




def _to_numpy(x):
    if hasattr(x, "detach"):
        x = x.detach().cpu().numpy()
    return np.asarray(x)


def l2_normalize(X, eps=1e-8):
    X = _to_numpy(X).astype(np.float64)
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)


def cosine_distance_matrix(X, Y=None):
    X = l2_normalize(X)
    if Y is None:
        Y = X
    else:
        Y = l2_normalize(Y)
    return 1.0 - X @ Y.T


def retrieval_from_similarity(sim, true_indices=None):
    """
    sim: [n_queries, n_targets], larger is better.
    true_indices: target index for each query. Defaults to diagonal matching.
    """
    n = sim.shape[0]
    if true_indices is None:
        true_indices = np.arange(n)

    order = np.argsort(-sim, axis=1)
    ranks = np.empty(n, dtype=np.int64)

    for i, true_idx in enumerate(true_indices):
        ranks[i] = np.where(order[i] == true_idx)[0][0] + 1

    return {
        "mean_rank": float(ranks.mean()),
        "median_rank": float(np.median(ranks)),
        "r1": float(np.mean(ranks == 1)),
        "r5": float(np.mean(ranks <= 5)),
        "r10": float(np.mean(ranks <= 10)),
        "ranks": ranks,
    }


def retrieval_from_embeddings(query, target):
    query = l2_normalize(query)
    target = l2_normalize(target)
    sim = query @ target.T
    return retrieval_from_similarity(sim)


def rsa_between_spaces(X, Y, metric="cosine"):
    """
    Pearson correlation between vectorized upper triangles of RDMs.
    """
    X = _to_numpy(X)
    Y = _to_numpy(Y)

    DX = sp.spatial.distance.pdist(X, metric=metric)
    DY = sp.spatial.distance.pdist(Y, metric=metric)

    return float(np.corrcoef(DX, DY)[0, 1])


def hungarian_accuracy_from_plan(P):
    """
    One-to-one matching accuracy using the OT coupling as affinity.
    """
    row_ind, col_ind = linear_sum_assignment(-P)
    pred = np.empty(P.shape[0], dtype=np.int64)
    pred[row_ind] = col_ind
    return float(np.mean(pred == np.arange(P.shape[0])))


def compute_ot_baseline(
    Zs,
    Zt,
    method="sinkhorn",
    reg=5e-2,
    gw_reg=5e-3,
    normalize=True,
    max_iter=10_000,
    tol=1e-9,
):
    """
    Compute OT/GW baseline between two embedding spaces.

    Parameters
    ----------
    Zs, Zt : array-like, shape [n_images, d]
        Source and target embeddings, sorted in the same image order.
    method : {"emd", "sinkhorn", "gw", "entropic_gw"}
        OT method.
    reg : float
        Entropic regularization for Sinkhorn.
    gw_reg : float
        Entropic regularization for entropic GW.
    normalize : bool
        Whether to L2-normalize embeddings before distances/retrieval.

    Returns
    -------
    dict with:
        - plan metrics: retrieval directly from OT coupling P
        - barycentric metrics: retrieval after barycentric projection into target space
        - rsa: geometry similarity between original spaces
    """
    Zs = _to_numpy(Zs)
    Zt = _to_numpy(Zt)

    assert Zs.shape[0] == Zt.shape[0], "Zs and Zt must have same number of images."
    n = Zs.shape[0]

    if normalize:
        Zs_eval = l2_normalize(Zs)
        Zt_eval = l2_normalize(Zt)
    else:
        Zs_eval = Zs.astype(np.float64)
        Zt_eval = Zt.astype(np.float64)

    a = np.ones(n, dtype=np.float64) / n
    b = np.ones(n, dtype=np.float64) / n

    method = method.lower()

    if method in {"emd", "sinkhorn"}:
        # Direct OT cost between source and target embeddings.
        M = cosine_distance_matrix(Zs_eval, Zt_eval)
        M = M / (M.mean() + 1e-12)

        if method == "emd":
            P = ot.emd(a, b, M)
        else:
            P = ot.sinkhorn(
                a,
                b,
                M,
                reg=reg,
                numItermax=max_iter,
                stopThr=tol,
                verbose=False,
            )

    elif method in {"gw", "gromov", "entropic_gw", "entropic-gw"}:
        # GW uses within-space geometry only.
        C1 = cosine_distance_matrix(Zs_eval)
        C2 = cosine_distance_matrix(Zt_eval)
        C1 = C1 / (C1.mean() + 1e-12)
        C2 = C2 / (C2.mean() + 1e-12)

        if method in {"gw", "gromov"}:
            P = ot.gromov.gromov_wasserstein(
                C1,
                C2,
                a,
                b,
                loss_fun="square_loss",
                max_iter=max_iter,
                tol=tol,
                verbose=False,
            )
        else:
            P = ot.gromov.entropic_gromov_wasserstein(
                C1,
                C2,
                a,
                b,
                loss_fun="square_loss",
                epsilon=gw_reg,
                max_iter=max_iter,
                tol=tol,
                verbose=False,
            )
    else:
        raise ValueError(f"Unknown method: {method}")

    # 1) Direct matching from coupling matrix.
    # Larger coupling = stronger match.
    plan_metrics = retrieval_from_similarity(P)

    # 2) Barycentric projection into target space.
    # Since rows of P sum to 1/n, divide by row mass.
    row_mass = P.sum(axis=1, keepdims=True)
    Zs_bary = (P @ Zt_eval) / (row_mass + 1e-12)

    bary_metrics = retrieval_from_embeddings(Zs_bary, Zt_eval)

    # Cosine to correct target after barycentric mapping.
    diag_cos = np.sum(l2_normalize(Zs_bary) * l2_normalize(Zt_eval), axis=1)

    return {
        "method": method,
        "P": P,
        "plan_mean_rank": plan_metrics["mean_rank"],
        "plan_r1": plan_metrics["r1"],
        "plan_r5": plan_metrics["r5"],
        "plan_hungarian_acc": hungarian_accuracy_from_plan(P),
        "bary_mean_rank": bary_metrics["mean_rank"],
        "bary_median_rank": bary_metrics["median_rank"],
        "bary_r1": bary_metrics["r1"],
        "bary_r5": bary_metrics["r5"],
        "bary_r10": bary_metrics["r10"],
        "bary_cosine": float(diag_cos.mean()),
        "rsa": rsa_between_spaces(Zs_eval, Zt_eval),
    }

In [3]:
template_file = "../scripts/mlp_embeddings_standarized/avg_ws_mlp_v1_128_768_sub-{subject:02d}.pt"

encoder_subject = []

for subject in trange(1, 9):
    pck = torch.load(template_file.format(subject=subject))
    Z_test = pck["Z_test"]
    Z_test = Z_test[:515, :] # Already sorted
    encoder_subject.append(Z_test)

  0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
Zs = encoder_subject[0]  # S1
Zt = encoder_subject[1]  # S2

res_sinkhorn = compute_ot_baseline(Zs, Zt, method="sinkhorn", reg=5e-2)
res_gw = compute_ot_baseline(Zs, Zt, method="entropic_gw", gw_reg=5e-3)


print(res_gw)

{'method': 'sinkhorn', 'P': array([[1.22358313e-04, 4.04590951e-06, 1.50515662e-09, ...,
        1.92328752e-08, 3.41151483e-08, 2.97758631e-07],
       [8.71417007e-05, 1.08359869e-05, 2.77467159e-07, ...,
        4.08142306e-08, 3.50039910e-08, 2.58852984e-06],
       [2.56495064e-07, 2.07068642e-06, 2.05414514e-06, ...,
        1.58059346e-08, 3.13848051e-06, 2.55232317e-08],
       ...,
       [6.42042555e-07, 1.79181044e-07, 7.39961430e-08, ...,
        1.78889035e-06, 2.79909623e-05, 2.11713860e-08],
       [8.83820229e-06, 5.37988498e-07, 5.83547647e-10, ...,
        3.43676664e-08, 2.14293681e-07, 4.54878766e-08],
       [7.40531449e-08, 1.97126581e-07, 6.08384674e-08, ...,
        1.25853966e-06, 1.90158590e-06, 9.29656833e-08]], shape=(515, 515)), 'plan_mean_rank': 153.35533980582525, 'plan_r1': 0.013592233009708738, 'plan_r5': 0.05242718446601942, 'plan_hungarian_acc': 0.007766990291262136, 'bary_mean_rank': 142.04660194174758, 'bary_median_rank': 89.0, 'bary_r1': 0.01941747

In [15]:
from tqdm.notebook import tqdm
import pandas as pd

def run_ot_all_pairs(encoder_subject, methods=("sinkhorn", "entropic_gw")):
    rows = []

    for method in methods:
        for s in tqdm(range(8), desc=method):
            for t in trange(8, leave=False):
                if s == t:
                    continue

                res = compute_ot_baseline(
                    encoder_subject[s],
                    encoder_subject[t],
                    method=method,
                    reg=5e-2,
                    gw_reg=5e-3,
                )

                rows.append({
                    "method": method,
                    "source": s + 1,
                    "target": t + 1,
                    "mean_rank": res["bary_mean_rank"],
                    "r1": res["bary_r1"],
                    "r5": res["bary_r5"],
                    "cosine": res["bary_cosine"],
                    "rsa": res["rsa"],
                    "plan_mean_rank": res["plan_mean_rank"],
                    "plan_r1": res["plan_r1"],
                    "hungarian_acc": res["plan_hungarian_acc"],
                })

    return pd.DataFrame(rows)


df_ot = run_ot_all_pairs(
    encoder_subject,
    methods=("emd", "sinkhorn", "entropic_gw"),
)

df_ot.groupby("method")[["mean_rank", "r1", "r5", "cosine", "rsa", "plan_mean_rank", "plan_r1"]].agg(["mean", "std"])

emd:   0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

...

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

mean_rank                   r1                  r5            \
                   mean        std      mean       std      mean       std   
method                                                                       
emd          223.915846  50.742009  0.006241  0.005298  0.023024  0.015945   
entropic_gw   80.961928  97.428622  0.464528  0.405199  0.546463  0.430420   
sinkhorn     207.201907  75.636414  0.007351  0.006805  0.033010  0.025616   

               cosine                 rsa           plan_mean_rank             \
                 mean       std      mean       std           mean        std   
method                                                                          
emd          0.053115  0.064652  0.634677  0.038999     256.961026   1.201238   
entropic_gw  0.561698  0.368090  0.634677  0.038999      71.996567  92.466008   
sinkhorn     0.095907  0.113887  0.634677  0.038999     212.237032  68.416656   

              plan_r1            
                 mean       std  
method                           
emd          0.006241  0.005298  
entropic_gw  0.480964  0.412399  
sinkhorn     0.006172  0.005832

In [18]:
df_ot.to_parquet("ot_alignment_results-encoder.parquet")
df_ot.sort_values("mean_rank")


,method,source,target,mean_rank,r1,r5,cosine,rsa,plan_mean_rank,plan_r1,hungarian_acc
119,entropic_gw,2,1,1.000000,1.000000,1.000000,0.991884,0.708111,1.000000,1.000000,1.000000
112,entropic_gw,1,2,1.000000,1.000000,1.000000,0.991967,0.708111,1.000000,1.000000,1.000000
131,entropic_gw,3,7,1.499029,0.899029,0.972816,0.930741,0.689013,1.281553,0.904854,0.935922
128,entropic_gw,3,4,1.712621,0.899029,0.966990,0.925041,0.673213,1.248544,0.912621,0.947573
156,entropic_gw,7,3,1.829126,0.885437,0.966990,0.922032,0.689013,1.281553,0.910680,0.935922
...,...,...,...,...,...,...,...,...,...,...,...
93,sinkhorn,6,3,341.157282,0.000000,0.003883,-0.099811,0.644230,329.201942,0.000000,0.000000
92,sinkhorn,6,2,350.720388,0.000000,0.000000,-0.096298,0.571162,342.798058,0.000000,0.000000
67,sinkhorn,2,6,365.130097,0.000000,0.001942,-0.130423,0.571162,343.965049,0.000000,0.000000
97,sinkhorn,6,8,368.502913,0.001942,0.001942,-0.141332,0.604510,366.178641,0.001942,0.001942


In [25]:
def raw_fmri_avg(subject: int, template_file: str):
    df_reps = split_repetitions(subject=subject)
    df_reps = df_reps.query("nsd_id in @test_stimuli").reset_index(drop=True)


    X = np.load(template_file.format(subject=subject))

    X1 = X[df_reps.subject_index_1]
    X2 = X[df_reps.subject_index_2]
    X3 = X[df_reps.subject_index_3]
    X_avg = (X1 + X2 + X3) / 3

    return X_avg

template_file = "../scripts/steps/subject-{subject:02d}_betas_prec.npy"
template_file = "../scripts/steps/subject-{subject:02d}_betas_pca-no-rel.npy"
raw_fmri = []

for subject in trange(1, 9):
    raw_fmri.append(torch.tensor(raw_fmri_avg(subject, template_file)))

  0%|          | 0/8 [00:00<?, ?it/s]

In [26]:
raw_fmri[0].shape

torch.Size([515, 768])

In [27]:


df_ot = run_ot_all_pairs(
    raw_fmri,
    methods=("emd", "sinkhorn", "entropic_gw"),
)

df_ot.to_parquet("ot_alignment_results-fmri-pca-no-rel.parquet")
df_ot.groupby("method")[["mean_rank", "r1", "r5", "cosine", "rsa", "plan_mean_rank", "plan_r1"]].agg(["mean", "std"])

emd:   0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

...

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

mean_rank                   r1                  r5            \
                   mean        std      mean       std      mean       std   
method                                                                       
emd          197.925208  35.566290  0.007490  0.006492  0.026456  0.016495   
entropic_gw  166.046394  56.109037  0.011616  0.010378  0.046186  0.035353   
sinkhorn     188.737067  41.207300  0.007108  0.006427  0.028294  0.018113   

               cosine                 rsa           plan_mean_rank             \
                 mean       std      mean       std           mean        std   
method                                                                          
emd          0.106578  0.056889  0.390711  0.082251     256.720042   1.656369   
entropic_gw  0.222018  0.121787  0.390711  0.082251     169.871741  47.651902   
sinkhorn     0.172407  0.090334  0.390711  0.082251     187.263003  40.435032   

              plan_r1            
                 mean       std  
method                           
emd          0.007490  0.006492  
entropic_gw  0.012309  0.011687  
sinkhorn     0.006900  0.006434

In [28]:

template_file = "../scripts/steps/subject-{subject:02d}_betas_pca.npy"
raw_fmri = []

for subject in trange(1, 9):
    raw_fmri.append(torch.tensor(raw_fmri_avg(subject, template_file)))


df_ot = run_ot_all_pairs(
    raw_fmri,
    methods=("emd", "sinkhorn", "entropic_gw"),
)

df_ot.to_parquet("ot_alignment_results-fmri-pca.parquet")
df_ot.groupby("method")[["mean_rank", "r1", "r5", "cosine", "rsa", "plan_mean_rank", "plan_r1"]].agg(["mean", "std"])

  0%|          | 0/8 [00:00<?, ?it/s]

emd:   0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

...

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

  0%|          | 0/8 [00:00<?, ?it/s]

mean_rank                   r1                  r5            \
                   mean        std      mean       std      mean       std   
method                                                                       
emd          175.302323  43.367390  0.008669  0.006272  0.032420  0.016296   
entropic_gw  215.618343  92.329129  0.009397  0.011661  0.034778  0.037877   
sinkhorn     164.625763  48.212148  0.008703  0.005417  0.037240  0.020149   

               cosine                 rsa          plan_mean_rank             \
                 mean       std      mean      std           mean        std   
method                                                                         
emd          0.175574  0.089388  0.531648  0.06646     256.377843   1.577456   
entropic_gw  0.120576  0.245330  0.531648  0.06646     214.757906  97.139014   
sinkhorn     0.253393  0.125216  0.531648  0.06646     165.536755  48.281470   

              plan_r1            
                 mean       std  
method                           
emd          0.008669  0.006272  
entropic_gw  0.010194  0.012271  
sinkhorn     0.008287  0.005016